# VPA Demo: Direct Tool Runs

Use this notebook when you want to run one tool at a time and manually
decide what to do next. The main deep-learning example is ProteinMPNN
sequence design on a fixture backbone, followed by lightweight analysis
and QC commands. Structure prediction with Protenix-v2 is shown as the
next manual step.

By default, this notebook is smoke-safe: it checks GPU/CLI visibility and
prints the heavy commands instead of launching model jobs. Set
`VPA_DEMO_RUN_HEAVY=1` before executing the notebook to actually start
workers and run ProteinMPNN/Protenix-v2.

In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
import textwrap
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "vibeproteinagent").exists():
            return candidate
    return Path("/opt/vpa-data/VibeProteinAgent")

REPO_ROOT = find_repo_root()
VPA = os.environ.get("VPA_BIN", str(Path("/opt/miniconda3/envs/vpa/bin/vpa")))
if not Path(VPA).exists():
    VPA = os.environ.get("VPA_BIN", "vpa")

RUN_HEAVY = os.environ.get("VPA_DEMO_RUN_HEAVY", "0") == "1"
RUN_LLM = os.environ.get("VPA_DEMO_RUN_LLM", "0") == "1"
RUN_INSTALL = os.environ.get("VPA_DEMO_RUN_INSTALL", "0") == "1"
WORKSPACE_PREFIX = os.environ.get("VPA_DEMO_WORKSPACE_PREFIX", "demo")

print(f"repo: {REPO_ROOT}")
print(f"vpa: {VPA}")
print(f"RUN_INSTALL={RUN_INSTALL} RUN_HEAVY={RUN_HEAVY} RUN_LLM={RUN_LLM}")


def run(cmd, *, check=True, cwd=REPO_ROOT, env=None, timeout=None):
    if isinstance(cmd, str):
        printable = cmd
        shell = True
    else:
        printable = " ".join(shlex.quote(str(part)) for part in cmd)
        shell = False
    print(f"$ {printable}")
    proc = subprocess.run(
        cmd,
        shell=shell,
        cwd=str(cwd),
        env={**os.environ, **(env or {})},
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
    )
    if proc.stdout:
        print(proc.stdout[-6000:])
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {printable}")
    return proc


def show_command(parts):
    print(" ".join(shlex.quote(str(part)) for part in parts))


## Optional Install For This Notebook

Run this cell only when the selected direct-run tools are not installed yet. This notebook does not use ESM3, so no Hugging Face credential is needed for the shown path.


In [ ]:
install_cmd = [
    "npx", "vibeproteinagent", "install",
    "--tools", "proteinmpnn,protenix_v2,protein_qc,sequence_analysis",
    "--cuda-profile", "auto",
    "--yes",
]
if RUN_INSTALL:
    run(install_cmd, timeout=7200)
else:
    show_command(install_cmd)


## 1. GPU And CLI Smoke

In [ ]:
# This cell is intentionally lightweight. It proves the notebook is running on a
# GPU-capable node and that PyTorch can see CUDA. The heavy VPA jobs below are
# gated behind RUN_HEAVY so the tutorial remains safe to execute.
run("hostname")
run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader | head -3", check=False)
import sys
run([sys.executable, "-c", "import torch; print('torch', torch.__version__); print('cuda_available', torch.cuda.is_available()); print('device_count', torch.cuda.device_count()); print('device0', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')"])

## 2. Inspect The Tools You Will Use

In [ ]:
for tool in ["proteinmpnn", "protenix_v2", "protein_qc", "sequence_analysis"]:
    run([VPA, "tool", "info", tool, "--json"], check=False, timeout=60)

## 3. Start A Workspace And Worker

In [ ]:
workspace = f"{WORKSPACE_PREFIX}_direct_tools"
if RUN_HEAVY:
    run([VPA, "daemon", "start", "--workers", "1", "--workspace", workspace], timeout=120)
    run([VPA, "daemon", "status", "--workspace", workspace], check=False, timeout=60)
else:
    show_command([VPA, "daemon", "start", "--workers", "1", "--workspace", workspace])

## 4. Run ProteinMPNN, Then Analysis/QC

In [ ]:
workspace = f"{WORKSPACE_PREFIX}_direct_tools"
backbone = REPO_ROOT / "tests" / "fixtures" / "6MRR.pdb"
proteinmpnn_cmd = [
    VPA, "run", "proteinmpnn",
    f"pdb_path={backbone}",
    "num_designs=2",
    "temperature=0.1",
    "seed=17",
    "--workspace", workspace,
    "--follow",
    "--json",
]
qc_cmd = [
    VPA, "run", "protein_qc",
    "sequence=ACDEFGHIKLMNPQRSTVWY",
    "--workspace", workspace,
    "--follow",
    "--json",
]
analysis_cmd = [
    VPA, "run", "sequence_analysis",
    "sequence=ACDEFGHIKLMNPQRSTVWY",
    "--workspace", workspace,
    "--json",
]

if RUN_HEAVY:
    run(proteinmpnn_cmd, timeout=900)
    run(qc_cmd, timeout=300)
    run(analysis_cmd, timeout=300)
else:
    show_command(proteinmpnn_cmd)
    show_command(qc_cmd)
    show_command(analysis_cmd)

## 5. Manual Structure Prediction Step

In [ ]:
workspace = f"{WORKSPACE_PREFIX}_direct_tools"
protenix_input = REPO_ROOT / "tests" / "fixtures" / "protenix_v2_input_tiny.json"
protenix_cmd = [
    VPA, "run", "protenix_v2",
    f"input_json={protenix_input}",
    "seeds=[101]",
    "use_msa=false",
    "n_sample=1",
    "n_step=20",
    "n_cycle=1",
    "--workspace", workspace,
    "--follow",
    "--json",
]
if RUN_HEAVY:
    run(protenix_cmd, timeout=1800)
else:
    show_command(protenix_cmd)

## 6. Find Outputs And Make A Human Report

In [ ]:
workspace = f"{WORKSPACE_PREFIX}_direct_tools"
for parts in [
    [VPA, "status", "<job_id>", "--workspace", workspace],
    [VPA, "wait", "<job_id>", "--workspace", workspace],
    [VPA, "results", "<job_id>", "--workspace", workspace, "--json"],
    [VPA, "artifacts", "<job_id>", "--workspace", workspace],
    [VPA, "report", "<job_id>", "--workspace", workspace, "--open"],
]:
    show_command(parts)